In [31]:
import pandas as pd

df = pd.read_csv("data/raw_weather/INMET_SE_MG_A515_VARGINHA_13-07-2006_A_31-12-2006.CSV", sep=';')

# limpar nomes das colunas
df.columns = df.columns.str.strip()

df.head()

,DATA (YYYY-MM-DD),HORA (UTC),"PRECIPITA��O TOTAL, HOR�RIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESS�O ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESS�O ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m�),"TEMPERATURA DO AR - BULBO SECO, HORARIA (�C)",TEMPERATURA DO PONTO DE ORVALHO (�C),TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C),TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (�C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (�C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRE��O HORARIA (gr) (� (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2006-07-13,00:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN
1,2006-07-13,01:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN
2,2006-07-13,02:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN
3,2006-07-13,03:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN
4,2006-07-13,04:00,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,-9999,NaN


In [32]:
df = df[[
    'DATA (YYYY-MM-DD)',
    'HORA (UTC)',
    'PRECIPITA��O TOTAL, HOR�RIO (mm)',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (�C)',
    'TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C)',
    'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)'
]]

In [33]:
df = df.rename(columns={
    'DATA (YYYY-MM-DD)': 'data',
    'HORA (UTC)': 'hora',
    'PRECIPITA��O TOTAL, HOR�RIO (mm)': 'chuva',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (�C)': 'temp',
    'TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C)': 'temp_max',
    'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)': 'temp_min',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade'
})

df

,data,hora,chuva,temp,temp_max,temp_min,umidade
0,2006-07-13,00:00,-9999,-9999,-9999,-9999,-9999
1,2006-07-13,01:00,-9999,-9999,-9999,-9999,-9999
2,2006-07-13,02:00,-9999,-9999,-9999,-9999,-9999
3,2006-07-13,03:00,-9999,-9999,-9999,-9999,-9999
4,2006-07-13,04:00,-9999,-9999,-9999,-9999,-9999
...,...,...,...,...,...,...,...
4123,2006-12-31,19:00,0,22,"22,1","21,5",80
4124,2006-12-31,20:00,0,"21,5","22,1","21,5",82
4125,2006-12-31,21:00,0,"20,2","21,6","20,2",96
4126,2006-12-31,22:00,0,"19,1","20,2","19,1",93


In [34]:
df['datetime'] = pd.to_datetime(df['data'] + ' ' + df['hora'])
df = df.set_index('datetime')

df

,data,hora,chuva,temp,temp_max,temp_min,umidade
datetime,,,,,,,
2006-07-13 00:00:00,2006-07-13,00:00,-9999,-9999,-9999,-9999,-9999
2006-07-13 01:00:00,2006-07-13,01:00,-9999,-9999,-9999,-9999,-9999
2006-07-13 02:00:00,2006-07-13,02:00,-9999,-9999,-9999,-9999,-9999
2006-07-13 03:00:00,2006-07-13,03:00,-9999,-9999,-9999,-9999,-9999
2006-07-13 04:00:00,2006-07-13,04:00,-9999,-9999,-9999,-9999,-9999
...,...,...,...,...,...,...,...
2006-12-31 19:00:00,2006-12-31,19:00,0,22,"22,1","21,5",80
2006-12-31 20:00:00,2006-12-31,20:00,0,"21,5","22,1","21,5",82
2006-12-31 21:00:00,2006-12-31,21:00,0,"20,2","21,6","20,2",96


In [35]:
for col in ['chuva', 'temp', 'temp_max', 'temp_min', 'umidade']:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '.')
        .astype(float)
    )

In [36]:
df = df.replace([-9999, -9999.0], pd.NA)
df = df.dropna()

In [37]:
df_daily = df.resample('D').agg({
    'chuva': 'sum',
    'temp': 'mean',
    'temp_max': 'max',
    'temp_min': 'min',
    'umidade': 'mean'
})

df_daily

,chuva,temp,temp_max,temp_min,umidade
datetime,,,,,
2006-07-13,0.0,21.25,24.7,16.6,48.25
2006-07-14,0.0,17.254167,24.1,10.5,58.791667
2006-07-15,0.0,16.054167,24.1,7.9,56.958333
2006-07-16,0.0,16.1625,22.4,9.3,58.333333
2006-07-17,0.0,15.804167,23.3,8.7,66.416667
...,...,...,...,...,...
2006-12-27,0.2,22.6125,29.9,19.2,79.333333
2006-12-28,0.2,21.991667,28.4,18.3,79.75
2006-12-29,0.2,22.283333,28.2,18.7,73.583333
